# 1. Data Cleaning - Akkadian to English Translation
## Aaron Dichoso & Luis Razon

This notebook details the steps performed for cleaning the dataset used in the Deep Past Challenge for Translating Akkadian Text to English.
The competition can be accessed in this link: https://www.kaggle.com/competitions/deep-past-initiative-machine-translation/data

Run this notebook BEFORE all other notebooks.

In [ ]:
import pandas as pd
import numpy as np
import re

#Import data
train_df = pd.read_csv("dataset/train.csv")
train_df.sample(5)

,oare_id,transliteration,translation
671,6866aa1e-6750-45b7-9100-8b5413832f53,a-na a-šur-SIG₅ qí-bi-ma um-ma ni-ni-ma a-na K...,To Aššur-damiq from Nini: I keep going for ......
890,8d328800-b327-4cae-9821-00b7e19ad967,a-na PUZUR₄-a-na a-lá-ḫi-im ù ma-nu-ki-a-šur a...,"To Puzur-Anna, Ali-ahum and Mannum-kī-Aššur; s..."
906,9006441d-52a5-4541-9980-272ef4f85f9c,a-na a-la-ḫi-im DUMU šál-ma-a-šur KIŠIB PUZUR₄...,"To Ali-ahum son of [Šalim-Aššur], seal of Puzu..."
1348,da4b9647-bf01-4d97-9c3d-75d26535ddbc,3 GÍN KÙ.BABBAR a-na e-na-aḫ-DINGIR 0.25 GÍN a...,3 shekels of silver for Enah-ilī;
1398,e3e3edd7-e12c-47fd-b637-f431e46f0b52,um-ma bé-li-a-ma a-na en-um-a-šur ù lá-qé-ep a...,From Bēliya to Ennam-Aššur and Lā-qēp; specifi...


Translation Dictionaries and regular expressions were used to achieve clean data on the given dataset.

1. Subscripts: Akkadian Transliterations had their subscripts replaced with the regular size character counterparts.
2. Special letters: From the dataset, it was mentioned that there was only one type of H in Akkadian, so this can be replaced with our H in english instead.
3. Special Characters: Certain special characters from the Akkadian transliterations and English translations were removed as these may be artifacts from the OCR software used.

Note: Some special characters were not removed in English as some of these characters actually hold valuable meaning.

(Ex. The punctuation marks actually being correctly used.)

In [ ]:
trans_subscripts = str.maketrans('₀₁₂₃₄₅₆₇₈₉ₓ', '0123456789x')
trans_h   = str.maketrans('ḫḪ', 'hH') #From dataset description: Only one type of H in akkadian, so this can be one-to-one.
trans_specials = str.maketrans('', '', '!?"——<>⌈⌋⌊[]+ʾ/;') # Special Characters
trans_specials_train = str.maketrans('', '', '<>⌈⌋⌊[]/;"') # Special Characters

Rational Numbers also had different representations across the dataset. First, floating point precision errors (Ex. 0.33333000000001) were MANUALLY cleaned (We found this easier by just using find+replace). Then, we used dictionaries to convert decimal numbers and unicode fractions to plain character representations of the fractions.

In [ ]:
DECIMAL_TO_FRAC = {
    r'0\.3{1,5}': "1/3",
    r'0\.6{4,5}': "2/3",
    r'0\.16{3,4}': "1/6",
    r'0\.83{3,4}': "5/6",
    r'0\.5': "1/2",
    r'0\.25': "1/4",
    r'0\.75': "3/4",
    r'0\.125': "1/8",
    r'0\.375': "3/8",
    r'0\.625': "5/8",
    r'0\.875': "7/8",
}

UNICODE_FRACTIONS = {
    "½": "1/2",
    "⅓": "1/3",
    "⅔": "2/3",
    "¼": "1/4",
    "¾": "3/4",
    "⅕": "1/5",
    "⅖": "2/5",
    "⅗": "3/5",
    "⅘": "4/5",
    "⅙": "1/6",
    "⅚": "5/6",
    "⅛": "1/8",
    "⅜": "3/8",
    "⅝": "5/8",
    "⅞": "7/8",
}

def convert_fractions(s):
    #Look for decimals to turn to fractions
    #Separate mixed numbers first (whole.dec => whole 0.dec)
    s = s.str.replace(re.compile(r'([0-9]|\s)\.([0-9]+)'), r'\1 0.\2', regex=True)

    #Look for decimals and convert into fractions (0.dec => num/den)
    for dec in DECIMAL_TO_FRAC.keys():
        s = s.str.replace(dec, DECIMAL_TO_FRAC[dec], regex=True)
    
    for unicode in UNICODE_FRACTIONS.keys():
        s = s.str.replace(unicode, UNICODE_FRACTIONS[unicode])
    
    #Remove lone zeroes just in case
    s = s.str.replace(" 0 ", " ")
    return s


Finally. we processed the Akkadian transliterations and English translations via a pipeline of several regular expressions applied on the series.

## Transliteration Cleaning Pipeline

In [4]:
def process_transliterations():
    s = train_df["transliteration"]

    #Replace colons -> spaces
    s = s.str.replace(r'\:', " ", regex=True) 
    
    #Remove unecessary periods (All except in numbers).
    s = s.str.replace(r'\s*\.\s', " ", regex=True) 
    s = s.str.replace(r' \.\s*', " ", regex=True)

    #Turn ellipsis (...) into <big_gap>
    s = s.str.replace(r'(\.{3,}|…+)', "<big_gap>", regex=True)

    # x -> <gap>
    s = s.str.replace('x', "<gap>", regex=False)

    #Subscripts and H cleaning
    s = s.str.translate(trans_subscripts)
    s = s.str.translate(trans_h)

    #Replace annotations for broken lines with gaps
    s = s.str.replace(r'\(broken line\)', '<gap>', regex=True)
    s = s.str.replace(r'\(large break|5 broken lines|4 broken lines|3 broken lines|2 broken lines|broken line\)', '<big_gap>', regex=True)

    #Remove special characters while still keeping the gaps (pad gaps with special characters so that they stay safe)
    #Protect special words inside parentheses so that we can remove the other parentheses

    s = s.str.replace('<gap>', '\x00gap\x00')
    s = s.str.replace('<big_gap>', '\x00big\x00')        
    s = s.str.translate(trans_specials)
    s = s.str.replace('\x00gap\x00', '<gap>')
    s = s.str.replace('\x00big\x00', '<big_gap>')

    #Convert all fractions
    s = convert_fractions(s)

    #Combine multiple gaps together to form a big gap
    s = s.str.replace(r'(<gap>(\s{0,}<gap>+)+)', "<big_gap>", regex=True)
    s = s.str.replace(r'(<big_gap>(\s{0,}<big_gap>+)+)', "<big_gap>", regex=True)
    s = s.str.replace(r'((\w[-]{0,1})<big_gap>(\s{0,}<big_gap>+)+)', r'\2<big_gap>', regex=True)
    s = s.str.replace(r"<gap>\s*<big_gap>", "<big_gap>", regex=True)
    s = s.str.replace(r"<big_gap>\s*<gap>", "<big_gap>", regex=True)
    
    s = s.str.replace(re.compile(r'\b(\w+)(?:\s+\1\b)+'), r'\1', regex=True) #Remove duplicate words next to each other
    s = s.str.replace(re.compile(r'\s+([.,:])'), r'\1', regex=True) #Remove long whitespaces before punctuations (             .)
    s = s.str.replace(re.compile(r'([.,])\1+'), r'\1', regex=True) #Remove duplicate punctations (,,,,,,,)

    #Remove duplicate whitespaces
    s = s.str.replace(r'\s+', ' ', regex=True) 

    #Remove unneeded parentheses while keeping those that have words in them
    s = s.str.replace(re.compile(r'\((mushen|TÚG|lu2|kush|gesh|dub|uru|na4|id2|kur|mi|HI|ki|e2|u2|m|d)\)'), r"[\1]", regex=True) 
    s = s.str.replace(r'\(|\)', "", regex=True) 
    s = s.str.replace(re.compile(r'\[(mushen|TÚG|lu2|kush|gesh|dub|uru|na4|id2|kur|mi|HI|ki|e2|u2|m|d)\]'), r"(\1)", regex=True) 
    
    #Last cleaning of white spaces
    s = s.str.strip()

    train_df["transliteration"] = s


## Translation Cleaning Pipeline

In [5]:
def process_translations():
    s = train_df["translation"]

    #Turn ellipsis (...) into <big_gap>
    s = s.str.replace(r'(\.{3,}|…+)', "<big_gap>", regex=True)
    s = s.str.replace(r'\s*x ', "<gap>", regex=True)

    #Subscripts and H cleaning
    s = s.str.translate(trans_subscripts)
    s = s.str.translate(trans_h)

    #Remove special characters while still keeping the gaps
    s = s.str.replace('<gap>', '\x00gap\x00')
    s = s.str.replace('<big_gap>', '\x00big\x00')

    s = s.str.replace(r'\(((\w+\s*)+)(\?|\!)\)', r'(\1)', regex=True) #Remove ? and ! inside parentheses with actual notes (note?)
    s = s.str.replace(r'\[((\w+\s*)+)(\?|\!)\]', r'[\1]', regex=True) #Remove ? and ! inside brackets with actual notes [note?]

    #Remove annotations inside parentheses
    regex_annotations = re.compile(r'\((fem|plur|pl|sing|singular|plural|\?|\!)\..*?\)', re.I)
    s = s.str.replace(regex_annotations, '', regex=True)
    s = s.str.replace(r'\((\?|\!)\)', '', regex=True) #Remove (?) and (!)
    s = s.str.replace(r'\s*(—|—|-) ', ' ', regex=True)
    s = s.str.replace(r' (—|—|-)(\w)', r' \2', regex=True)
    s = s.str.translate(trans_specials_train)
    s = s.str.replace('\x00gap\x00', '<gap>')
    s = s.str.replace('\x00big\x00', '<big_gap>')

    #Convert all fractions
    s = convert_fractions(s)
    
    #Combine multiple gaps together to form a big gap
    s = s.str.replace(r'(<gap>(\s{0,}<gap>+)+)', "<big_gap>", regex=True)
    s = s.str.replace(r'(<big_gap>(\s{0,}<big_gap>+)+)', "<big_gap>", regex=True)
    s = s.str.replace(r'((\w[-]{0,1})<big_gap>(\s{0,}<big_gap>+)+)', r'\2<big_gap>', regex=True)
    s = s.str.replace(r"<gap>\s*<big_gap>", "<big_gap>", regex=True)
    s = s.str.replace(r"<big_gap>\s*<gap>", "<big_gap>", regex=True)

    #Seperate words that are attached to gaps
    s = s.str.replace(r'(.)(<gap>|<big_gap>)(.)', r'\1 \2 \3', regex=True)

    s = s.str.replace(re.compile(r'\b(\w+)(?:\s+\1\b)+'), r'\1', regex=True) #Remove duplicate words next to each other
    s = s.str.replace(re.compile(r'\s+([.,:])'), r'\1', regex=True) #Remove long whitespaces before punctuations (             .)
    s = s.str.replace(re.compile(r'([.,])\1+'), r'\1', regex=True) #Remove duplicate punctations (,,,,,,,)

    #Remove duplicate whitespaces
    s = s.str.replace(r'\s+', ' ', regex=True) 

    #Remove parentheses
    s = s.str.replace(r'\(|\)', "", regex=True) 

    train_df["translation"] = s


In [6]:
process_transliterations()
process_translations()

train_df.sample(5)

,oare_id,transliteration,translation
999,9da3b494-b9f8-4eae-aebc-7dd4df44adb1,a-na a-lá-hi-im qí-bi-ma um-ma SIG5-pì-a-šur-m...,To Ali-ahum from Dami-pī-Aššur: With respect t...
1033,a5114979-d798-46a7-9943-6bba8ddf7382,um-ma <big_gap> ù lá-ma-sí-ma a-na <big_gap> a...,"From <big_gap> and Lamassī to Adida, Ababa and..."
12,0225bc1b-0bca-4fdc-820e-691e96f08e25,um-ma i-tur4-DINGIR-ma a-na en-um-a-šur ù a-lá...,From Itūr-ilī to Ennam-Aššur and Ali-ahum: Urg...
989,9b777791-39c9-4df0-9b8d-dea89df132e2,2 GÍN KÙ.BABBAR i-na li-bi4 ili5-me-ta-ak i-na...,2 shekels of silver owed by Ilī-wēdāku. He wil...
735,74cb2f0b-311a-41a8-b51d-23c93283dcd0,a-na a-šur-ni-im-ri ù a-lá-hi-im DUMU šál-ma-a...,To Aššur-nimrī and Ali-ahum son of Šalim-Aššur...


## Removing incomplete translations

Some entries in the dataset are very clearly incomplete. In these entries, the length of the transliteration far exceeds the length of the translation.

Example:

i-na 48 ku-ta-ni ša ik-ri-bi ŠÀ.BA 5 ku-ta-ni a-na 0.5 ma-na 7.5 GÍN ší-im-šu-nu a-na GAL pé-er-dí a-dí-in 7 ku-ta-ni a-šur-ma-lik DUMU i-na-a a-na 1 ma-na 13 GÍN KÙ.BABBAR il₅-qé 1 ku-ta-nam i-tur₄-DINGIR il₅-qé 1 ku-ta-nam a-na 7.5 GÍN KÙ.BABBAR ku-tù-bi-iš il₅-qé 3 ku-ta-ni ṣa-aḫ-ri-ì-lí il₅-qé 6 ku-ta-nu i-na li-bi₄ a-na-na 2 ku-ta-ni ḫa-té-e nu-lá-bi-iš 10 ku-ta-ni a-lu-ú-a ú-šé-ra-ba-am 1 ku-ta-nam SIG₅ a-na 0.5 ma-na KÙ.BABBAR tù-um-lá-i-um il₅-qé 5 ku-ta-ni a-na ḫa-ra-ni-a al-qé 6 ku-ta-ni a-na-kam e-zi-ib 1 ku-ta-nam a-na 13 GÍN en-na-nu-um

Translates to:

Of the 48 kutānus of the votive offerings: 5 kutānus I sold to the chief of mules for a price of 0.5 mina 7.5 shekels;

Even without knowing how to translate Akkadian, it is clearly seen that this is incomplete. Viewing the transliteration, there exists number values mentioned (6, 2, 10) That are not mentioned in the translation.

As we do not know where to cut the transliteration so that it corresponds to the translation, these are removed from the dataset.

In [7]:
#Isolate entries in the dataset where its translation is significantly shorter than its transliteration.
incomplete_df = train_df[train_df.apply(
    lambda r: 0.001 <= len(r.translation.split()) /
              max(len(r.transliteration.split()),1)
              <= 0.2,
    axis=1
)]

complete_df = train_df[~train_df.apply(tuple,1).isin(incomplete_df.apply(tuple,1))]

complete_df.to_csv("processed/cleaned_train_complete.csv", index=False)
incomplete_df.to_csv("processed/cleaned_train_incomplete.csv", index=False)

In [8]:
# Check sequence lengths to determine MAX_LEN
akk_lens = train_df["transliteration"].str.split().str.len()
eng_lens  = train_df["translation"].str.split().str.len()

print("Akkadian lengths:")
print(f"  Mean   : {akk_lens.mean():.1f}")
print(f"  Median : {akk_lens.median():.1f}")
print(f"  Max    : {akk_lens.max()}")
print(f"  95th % : {akk_lens.quantile(0.95):.1f}")

print("\nEnglish lengths:")
print(f"  Mean   : {eng_lens.mean():.1f}")
print(f"  Median : {eng_lens.median():.1f}")
print(f"  Max    : {eng_lens.max()}")
print(f"  95th % : {eng_lens.quantile(0.95):.1f}")

Akkadian lengths:
  Mean   : 55.9
  Median : 49.0
  Max    : 158
  95th % : 119.0

English lengths:
  Mean   : 90.0
  Median : 68.0
  Max    : 748
  95th % : 248.0


Data Cleaning ends here.